# Notebook 2.4f: Constrained Path Simulation

**Purpose:** Simulate student diversions constrained by real-world resource limits (candidate pool and ESC slots), then measure the contribution of existing vs hypothetical paths.

**Approach:**
1. Shuffle origin-ESC pairs randomly
2. Walk through in order, consuming raw mu against two coupled constraints
3. Record accepted amount per path
4. Repeat across N seeds (Monte Carlo)
5. Report mean and CI of existing vs hypothetical contribution

**Key Fixes from 2.4e:**
- Coupled depletion (both origin AND destination must have capacity)
- Raw mu values (no floor — preserves fractional predictions)
- Operates on unexploded data (325K pairs, not 5.28M)
- Candidate pool scoped to flows into congested schools only

---
# 0. Setup

In [1]:
import re
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm

pd.set_option("display.max_columns", None)

PROJECT_DIR = Path.cwd().parent
OUTPUT_DIR = PROJECT_DIR / "output"

print(f"Project directory: {PROJECT_DIR}")

Project directory: /workspace/project_paaral


---
# 1. Load Data

In [38]:
# Load school information
fpath = str(OUTPUT_DIR / "processed_project_bukas_school_information.parquet")
sch_info = pd.read_parquet(fpath)
print(f"\nSchool info shape: {sch_info.shape}")

# Create slim version with just the columns we need
essential_cols = [
    "school_id",
    "school_name",
    "old_region",
    "division",
    "sector",
    "offers_es",
    "offers_jhs",
    "offers_shs",
]
sch_info_slim = sch_info[essential_cols].copy()
sch_info_slim["school_id"] = sch_info_slim["school_id"].astype(str)

print(f"School info slim shape: {sch_info_slim.shape}")
print(f"\nSample:")
display(sch_info_slim.head())


School info shape: (61442, 20)
School info slim shape: (61442, 8)

Sample:


,school_id,school_name,old_region,division,sector,offers_es,offers_jhs,offers_shs
0,100001,Apaleng-Libtong ES,Region I,Ilocos Norte,Public,True,False,False
1,100002,Bacarra CES,Region I,Ilocos Norte,Public,True,False,False
2,100003,Buyon ES,Region I,Ilocos Norte,Public,True,False,False
3,100004,Ganagan Elementary School,Region I,Ilocos Norte,Public,True,False,False
4,100005,Macupit ES,Region I,Ilocos Norte,Public,True,False,False


In [2]:
# Candidate beneficiary pool
cbp = pd.read_parquet(OUTPUT_DIR / "full_candidate_beneficiary_pool_without_probdist_0207_model4.parquet")
cbp['origin_school_id'] = cbp['origin_school_id'].astype(str)
cbp['destination_school_id'] = cbp['destination_school_id'].astype(str)
print(f"Candidate beneficiary pool: {cbp.shape[0]:,} pairs")

Candidate beneficiary pool: 325,395 pairs


In [3]:
# Observed student flow
student_flow = pd.read_parquet(OUTPUT_DIR / "grade_7_student_flow_table_sy2324.parquet")
student_flow['school_id_origin'] = student_flow['school_id_origin'].astype(str)
student_flow['school_id_destination'] = student_flow['school_id_destination'].astype(str)
print(f"Observed student flow: {student_flow.shape[0]:,} pairs")

Observed student flow: 288,328 pairs


In [4]:
# Flow to congested public schools
flow_to_congested = pd.read_parquet(OUTPUT_DIR / "analysis_payload" / "flow_to_congested.parquet")
flow_to_congested['school_id_origin'] = flow_to_congested['school_id_origin'].astype(str)
print(f"Flow to congested: {flow_to_congested.shape[0]:,} rows")

Flow to congested: 36,164 rows


In [5]:
# ESC slot availability
esc_available = pd.read_parquet(OUTPUT_DIR / "analysis_payload" / "esc_available.parquet")
esc_available.rename(columns={'school_id': 'destination_school_id'}, inplace=True)
esc_available['destination_school_id'] = esc_available['destination_school_id'].astype(str)
esc_available.loc[esc_available['available_slots'] < 0, 'available_slots'] = 0

system_slots = esc_available['available_slots'].sum()
print(f"ESC schools: {esc_available.shape[0]:,}")
print(f"System available slots: {system_slots:,.0f}")

ESC schools: 1,447
System available slots: 26,608


---
# 2. Create `is_hypothetical` Flag

In [6]:
# Vectorized: create composite keys and check membership
existing_keys = set(
    student_flow['school_id_origin'] + '_' + student_flow['school_id_destination']
)
cbp_keys = cbp['origin_school_id'] + '_' + cbp['destination_school_id']
cbp['is_hypothetical'] = ~cbp_keys.isin(existing_keys)

print("Path classification:")
print(cbp['is_hypothetical'].value_counts().rename({True: 'Hypothetical', False: 'Existing'}))

Path classification:
is_hypothetical
Hypothetical    234972
Existing         90423
Name: count, dtype: int64


---
# 3. Build Resource Trackers

Two constraints:
1. **Candidate pool per origin** — scoped to non-beneficiaries flowing to congested schools only
2. **Available ESC slots per destination**

In [7]:
# Candidate pool: non-beneficiaries flowing to congested schools only
cand_pool = (
    flow_to_congested
    .groupby('school_id_origin')['count_non_beneficiary']
    .sum()
    .dropna()
    .to_dict()
)

total_candidates = sum(cand_pool.values())
print(f"Origins with candidates: {len(cand_pool):,}")
print(f"Total candidates (scoped to congested flows): {total_candidates:,.0f}")

Origins with candidates: 8,015
Total candidates (scoped to congested flows): 400,936


In [8]:
# ESC slots: only schools with available slots > 0
slot_pool = (
    esc_available[esc_available['available_slots'] > 0]
    .set_index('destination_school_id')['available_slots']
    .to_dict()
)

total_slots = sum(slot_pool.values())
print(f"ESC schools with slots: {len(slot_pool):,}")
print(f"Total available slots: {total_slots:,.0f}")

ESC schools with slots: 1,088
Total available slots: 26,491


---
# 4. Define Scenarios

In [9]:
mu_cols = [col for col in cbp.columns if re.search(r'mu_minus', col)]

SCENARIOS = {'baseline': 'mu_baseline_0_subsidy', 'current': 'mu_current_subsidy'}
SCENARIOS.update({f'minus_{n}k': f'mu_minus_{n}k_net_cost' for n in range(1, 21)})

print(f"{len(SCENARIOS)} scenarios defined.")

22 scenarios defined.


---
# 5. Simulation

For each iteration:
1. Shuffle 325K origin-ESC pairs
2. Walk through in order, consuming raw mu against coupled constraints
3. Record accepted amount per path
4. Aggregate by path type (existing vs hypothetical)

In [10]:
# Pre-extract static arrays (reused across all scenarios and iterations)
ARR_ORIGIN = cbp['origin_school_id'].values
ARR_DEST = cbp['destination_school_id'].values
ARR_HYPO = cbp['is_hypothetical'].values
N_PATHS = len(cbp)

def run_simulation(mu_values, cand_pool_init, slot_pool_init, seed):
    """
    Run one iteration of the constrained path simulation.
    
    Args:
        mu_values: numpy array of mu values for the chosen scenario
        cand_pool_init: Dict of origin_school_id -> candidate count (initial state)
        slot_pool_init: Dict of destination_school_id -> available slots (initial state)
        seed: Random seed for shuffling
    
    Returns:
        Dict with total_existing, total_hypothetical, total_both
    """
    # Copy trackers (fresh for each iteration)
    tracker_cand = dict(cand_pool_init)
    tracker_slots = dict(slot_pool_init)
    
    # Running totals for early termination
    remaining_cand = sum(cand_pool_init.values())
    remaining_slots = sum(slot_pool_init.values())
    
    # Shuffle indices (avoids DataFrame.sample overhead)
    rng = np.random.default_rng(seed)
    indices = np.arange(N_PATHS)
    rng.shuffle(indices)
    
    total_existing = 0.0
    total_hypothetical = 0.0
    
    for idx in indices:
        # Early termination if either resource is fully depleted
        if remaining_cand <= 0 or remaining_slots <= 0:
            break
        
        mu_val = mu_values[idx]
        if mu_val <= 0 or np.isnan(mu_val):
            continue
        
        og_id = ARR_ORIGIN[idx]
        dest_id = ARR_DEST[idx]
        
        current_pool = tracker_cand.get(og_id, 0)
        current_slots = tracker_slots.get(dest_id, 0)
        
        # Coupled depletion: both must have capacity
        if current_pool > 0 and current_slots > 0:
            accepted = min(current_pool, current_slots, mu_val)
            tracker_cand[og_id] -= accepted
            tracker_slots[dest_id] -= accepted
            remaining_cand -= accepted
            remaining_slots -= accepted
            
            if ARR_HYPO[idx]:
                total_hypothetical += accepted
            else:
                total_existing += accepted
    
    return {
        'total_existing': total_existing,
        'total_hypothetical': total_hypothetical,
        'total_both': total_existing + total_hypothetical,
        'remaining_candidates': remaining_cand,
        'remaining_slots': remaining_slots,
    }

print(f"Simulation function defined. Pre-extracted {N_PATHS:,} paths.")

Simulation function defined. Pre-extracted 325,395 paths.


---
# 6. Run Monte Carlo

In [11]:
# Test with a single iteration first
test_mu = cbp['mu_minus_1k_net_cost'].values

test_result = run_simulation(
    mu_values=test_mu,
    cand_pool_init=cand_pool,
    slot_pool_init=slot_pool,
    seed=42
)

print("=== Single Iteration Test (minus_1k) ===")
for k, v in test_result.items():
    print(f"  {k}: {v:,.1f}")

=== Single Iteration Test (minus_1k) ===
  total_existing: 11,789.0
  total_hypothetical: 14,150.8
  total_both: 25,939.8
  remaining_candidates: 374,996.2
  remaining_slots: 551.2


In [14]:
N_ITERATIONS = 100
SCENARIO_TO_RUN = 'mu_minus_1k_net_cost'  # Adjust as needed

# Extract mu array once for this scenario
mu_arr = cbp[SCENARIO_TO_RUN].values

results = []
for i in tqdm(range(N_ITERATIONS)):
    result = run_simulation(
        mu_values=mu_arr,
        cand_pool_init=cand_pool,
        slot_pool_init=slot_pool,
        seed=i
    )
    result['seed'] = i
    results.append(result)

mc_results = pd.DataFrame(results)
print(f"\nCompleted {N_ITERATIONS} iterations.")

100%|██████████| 100/100 [00:30<00:00,  3.27it/s]


Completed 100 iterations.


---
# 7. Analyze Results

In [15]:
# Summary statistics across iterations
print(f"=== Monte Carlo Results ({N_ITERATIONS} iterations, {SCENARIO_TO_RUN}) ===")
print(f"\nTotal diverted (both):")
print(f"  Mean: {mc_results['total_both'].mean():,.1f}")
print(f"  Std:  {mc_results['total_both'].std():,.1f}")
print(f"  Min:  {mc_results['total_both'].min():,.1f}")
print(f"  Max:  {mc_results['total_both'].max():,.1f}")

print(f"\nExisting paths:")
print(f"  Mean: {mc_results['total_existing'].mean():,.1f}")
print(f"  Std:  {mc_results['total_existing'].std():,.1f}")

print(f"\nHypothetical paths:")
print(f"  Mean: {mc_results['total_hypothetical'].mean():,.1f}")
print(f"  Std:  {mc_results['total_hypothetical'].std():,.1f}")

=== Monte Carlo Results (100 iterations, mu_minus_1k_net_cost) ===

Total diverted (both):
  Mean: 25,935.6
  Std:  10.5
  Min:  25,901.5
  Max:  25,958.4

Existing paths:
  Mean: 11,737.3
  Std:  82.6

Hypothetical paths:
  Mean: 14,198.3
  Std:  81.8


In [16]:
# Percentage from hypothetical paths
mc_results['pct_hypothetical'] = (
    mc_results['total_hypothetical'] / mc_results['total_both'] * 100
)

print(f"Hypothetical contribution:")
print(f"  Mean: {mc_results['pct_hypothetical'].mean():.1f}%")
print(f"  95% CI: [{mc_results['pct_hypothetical'].quantile(0.025):.1f}%, {mc_results['pct_hypothetical'].quantile(0.975):.1f}%]")

Hypothetical contribution:
  Mean: 54.7%
  95% CI: [54.1%, 55.3%]


In [17]:
# Remaining resources
print(f"\nRemaining resources after simulation:")
print(f"  Candidates: {mc_results['remaining_candidates'].mean():,.1f} (of {total_candidates:,.0f})")
print(f"  ESC slots:  {mc_results['remaining_slots'].mean():,.1f} (of {total_slots:,.0f})")


Remaining resources after simulation:
  Candidates: 375,000.4 (of 400,936)
  ESC slots:  555.4 (of 26,491)


---
# 8. Multi-Scenario Comparison (Optional)

Run the simulation across multiple subsidy scenarios to compare.

In [21]:
# Run across selected scenarios
SCENARIOS_TO_COMPARE = [
    'mu_current_subsidy',
    'mu_minus_1k_net_cost',
    'mu_minus_3k_net_cost',
    'mu_minus_5k_net_cost',
    'mu_minus_10k_net_cost',
    'mu_minus_15k_net_cost',
]

N_ITER_COMPARE = 100  # Fewer iterations for speed

scenario_summary = []

for scenario_col in tqdm(SCENARIOS_TO_COMPARE):
    # Extract mu array once per scenario
    mu_arr = cbp[scenario_col].values
    print(f"Running {scenario_col}...")
    
    iter_results = []
    for i in range(N_ITER_COMPARE):
        result = run_simulation(
            mu_values=mu_arr,
            cand_pool_init=cand_pool,
            slot_pool_init=slot_pool,
            seed=i
        )
        iter_results.append(result)
    
    df_iter = pd.DataFrame(iter_results)
    scenario_summary.append({
        'scenario': scenario_col,
        'mean_both': df_iter['total_both'].mean(),
        'mean_existing': df_iter['total_existing'].mean(),
        'mean_hypothetical': df_iter['total_hypothetical'].mean(),
        'pct_hypothetical': (df_iter['total_hypothetical'].mean() / df_iter['total_both'].mean() * 100),
        'mean_remaining_slots': df_iter['remaining_slots'].mean(),
    })

scenario_df = pd.DataFrame(scenario_summary)

print("\n=== Scenario Comparison ===")
display(scenario_df.style.format({
    'mean_both': '{:,.0f}',
    'mean_existing': '{:,.0f}',
    'mean_hypothetical': '{:,.0f}',
    'pct_hypothetical': '{:.2f}%',
    'mean_remaining_slots': '{:,.0f}',
}))

  0%|          | 0/6 [00:00<?, ?it/s]

Running mu_current_subsidy...


 17%|█▋        | 1/6 [00:31<02:35, 31.02s/it]

Running mu_minus_1k_net_cost...


 33%|███▎      | 2/6 [01:00<02:01, 30.31s/it]

...

 83%|████████▎ | 5/6 [02:28<00:29, 29.49s/it]

Running mu_minus_15k_net_cost...


100%|██████████| 6/6 [02:58<00:00, 29.69s/it]


=== Scenario Comparison ===


,scenario,mean_both,mean_existing,mean_hypothetical,pct_hypothetical,mean_remaining_slots
0,mu_current_subsidy,"25,931","11,734","14,196",55%,560
1,mu_minus_1k_net_cost,"25,936","11,737","14,198",55%,555
2,mu_minus_3k_net_cost,"25,945","11,745","14,200",55%,546
3,mu_minus_5k_net_cost,"25,951","11,750","14,201",55%,540
4,mu_minus_10k_net_cost,"25,962","11,758","14,204",55%,529
5,mu_minus_15k_net_cost,"25,971","11,764","14,207",55%,520


---
# 9. Export

In [ ]:
EXPORT_DIR = OUTPUT_DIR / "constrained_simulation"
EXPORT_DIR.mkdir(exist_ok=True)

mc_results.to_csv(EXPORT_DIR / "monte_carlo_results.csv", index=False)
print(f"Exported monte_carlo_results.csv")

scenario_df.to_csv(EXPORT_DIR / "scenario_comparison.csv", index=False)
print(f"Exported scenario_comparison.csv")

print(f"\nAll exports saved to: {EXPORT_DIR}")

---
# 10. ESC Schools with Highest Unmet Demand

**Purpose:** Identify which ESC schools would contribute most to decongestion if given additional slots.

**Approach:**
1. Filter CBP to origins that feed congested schools (exist in `cand_pool`)
2. Sum mu per destination ESC school — this is the total predicted demand
3. Compare against available slots
4. `unmet_demand = max(0, total_demand - available_slots)`
5. Rank ESC schools by unmet demand

In [45]:
# Filter CBP to origins that feed congested schools (i.e., exist in cand_pool)
congested_origins = set(cand_pool.keys())
cbp_congested = cbp[cbp['origin_school_id'].isin(congested_origins)].copy()
print(f"CBP pairs from congested-feeding origins: {cbp_congested.shape[0]:,} (of {cbp.shape[0]:,} total)")

# Sum mu per destination ESC school (using minus_1k as representative — scenarios are near-identical)
REPRESENTATIVE_SCENARIO = 'mu_minus_1k_net_cost'
demand_per_esc = (
    cbp_congested
    .groupby('destination_school_id')[REPRESENTATIVE_SCENARIO]
    .sum()
    .rename('total_predicted_demand')
    .reset_index()
)
print(f"Destination ESC schools with predicted demand: {demand_per_esc.shape[0]:,}")

CBP pairs from congested-feeding origins: 321,283 (of 325,395 total)
Destination ESC schools with predicted demand: 1,370


In [47]:
# Aggregate available slots per school (esc_available may have multiple rows per school)
esc_slots_agg = (
    esc_available
    .groupby('destination_school_id')['available_slots']
    .sum()
    .reset_index()
)
print(f"ESC schools before aggregation: {esc_available.shape[0]:,}")
print(f"ESC schools after aggregation:  {esc_slots_agg.shape[0]:,}")

# Merge with available slots
esc_demand = demand_per_esc.merge(
    esc_slots_agg,
    on='destination_school_id',
    how='left'
)
esc_demand['available_slots'] = esc_demand['available_slots'].fillna(0)

# Compute unmet demand
esc_demand['unmet_demand'] = (esc_demand['total_predicted_demand'] - esc_demand['available_slots']).clip(lower=0)

# Utilization ratio: how many times over are slots demanded?
esc_demand['demand_to_slot_ratio'] = (
    esc_demand['total_predicted_demand'] / esc_demand['available_slots'].replace(0, np.nan)
)

# Sort by unmet demand
esc_demand = esc_demand.sort_values('unmet_demand', ascending=False).reset_index(drop=True)

print(f"\nESC schools with unmet demand > 0: {(esc_demand['unmet_demand'] > 0).sum():,}")
print(f"Total unmet demand across system: {esc_demand['unmet_demand'].sum():,.0f}")
print(f"Total available slots: {esc_demand['available_slots'].sum():,.0f}")
print(f"Total predicted demand: {esc_demand['total_predicted_demand'].sum():,.0f}")

ESC schools before aggregation: 1,447
ESC schools after aggregation:  1,432

ESC schools with unmet demand > 0: 1,367
Total unmet demand across system: 833,433
Total available slots: 26,427
Total predicted demand: 859,801


In [48]:
# Top 10 ESC schools by unmet demand
print(f"=== Top 10 ESC Schools (of {(esc_demand['unmet_demand'] > 0).sum():,}) by Unmet Demand ===\n")
display(
    esc_demand.head(10).style.format({
        'total_predicted_demand': '{:,.0f}',
        'available_slots': '{:,.0f}',
        'unmet_demand': '{:,.0f}',
        'demand_to_slot_ratio': '{:.1f}x',
    })
)

=== Top 10 ESC Schools (of 1,367) by Unmet Demand ===



,destination_school_id,total_predicted_demand,available_slots,unmet_demand,demand_to_slot_ratio
0,407210,"9,896",71,"9,825",139.4x
1,406357,"4,848",47,"4,801",103.1x
2,406740,"4,489",36,"4,453",124.7x
3,406316,"4,404",32,"4,372",137.6x
4,406295,"4,156",6,"4,150",692.6x
5,406746,"4,158",30,"4,128",138.6x
6,406341,"4,058",45,"4,013",90.2x
7,482073,"4,058",56,"4,002",72.5x
8,406743,"3,937",38,"3,899",103.6x
9,406294,"3,849",24,"3,825",160.4x


In [49]:
# Distribution of unmet demand
print("=== Unmet Demand Distribution ===\n")
esc_with_unmet = esc_demand[esc_demand['unmet_demand'] > 0]

print(f"Schools with unmet demand: {len(esc_with_unmet):,}")
print(f"\nUnmet demand per school:")
print(f"  Mean:   {esc_with_unmet['unmet_demand'].mean():,.1f}")
print(f"  Median: {esc_with_unmet['unmet_demand'].median():,.1f}")
print(f"  Std:    {esc_with_unmet['unmet_demand'].std():,.1f}")
print(f"  Min:    {esc_with_unmet['unmet_demand'].min():,.1f}")
print(f"  Max:    {esc_with_unmet['unmet_demand'].max():,.1f}")

# Concentration: how many schools account for 50% of unmet demand?
cumulative = esc_with_unmet['unmet_demand'].cumsum() / esc_with_unmet['unmet_demand'].sum()
schools_for_50pct = (cumulative <= 0.5).sum() + 1
print(f"\nConcentration: Top {schools_for_50pct} schools account for 50% of total unmet demand")

=== Unmet Demand Distribution ===

Schools with unmet demand: 1,367

Unmet demand per school:
  Mean:   609.7
  Median: 344.2
  Std:    746.3
  Min:    2.3
  Max:    9,824.7

Concentration: Top 219 schools account for 50% of total unmet demand


In [50]:
# Link CBP pairs back to congested public JHS via shared origins
flow_to_congested['school_id_destination'] = flow_to_congested['school_id_destination'].astype(str)

esc_congested_link = (
    cbp_congested[['origin_school_id', 'destination_school_id']]
    .merge(
        flow_to_congested[['school_id_origin', 'school_id_destination']].rename(
            columns={
                'school_id_origin': 'origin_school_id',
                'school_id_destination': 'congested_school_id',
            }
        ),
        on='origin_school_id'
    )
)

# Count distinct congested public JHS each ESC school could help decongest
n_congested_destinations = (
    esc_congested_link
    .groupby('destination_school_id')['congested_school_id']
    .nunique()
    .rename('n_congested_destinations')
)

# How many unique feeder origins does each ESC school serve?
n_congested_origins = (
    cbp_congested
    .groupby('destination_school_id')['origin_school_id']
    .nunique()
    .rename('n_congested_origins')
)

# Hypothetical vs existing demand split per ESC school
path_type_demand = (
    cbp_congested
    .groupby(['destination_school_id', 'is_hypothetical'])[REPRESENTATIVE_SCENARIO]
    .sum()
    .unstack(fill_value=0)
)
# Rename using actual boolean index values (False=existing, True=hypothetical)
path_type_demand = path_type_demand.rename(columns={False: 'demand_existing', True: 'demand_hypothetical'})
# Ensure both columns exist
for col in ['demand_existing', 'demand_hypothetical']:
    if col not in path_type_demand.columns:
        path_type_demand[col] = 0.0

# Merge all enrichments into esc_demand
esc_demand = esc_demand.merge(n_congested_origins, left_on='destination_school_id', right_index=True, how='left')
esc_demand = esc_demand.merge(n_congested_destinations, left_on='destination_school_id', right_index=True, how='left')
esc_demand = esc_demand.merge(path_type_demand[['demand_existing', 'demand_hypothetical']], left_on='destination_school_id', right_index=True, how='left')
esc_demand['pct_demand_hypothetical'] = (
    esc_demand['demand_hypothetical'] / esc_demand['total_predicted_demand'] * 100
)

# Re-sort by unmet demand
esc_demand = esc_demand.sort_values('unmet_demand', ascending=False).reset_index(drop=True)

# Top 20 with enriched view
print("=== Top 20 ESC Schools — Enriched View ===\n")
display(
    esc_demand[['destination_school_id', 'total_predicted_demand', 'available_slots',
                'unmet_demand', 'demand_to_slot_ratio',
                'n_congested_origins', 'n_congested_destinations',
                'demand_existing', 'demand_hypothetical', 'pct_demand_hypothetical']]
    .head(20)
    .style.format({
        'total_predicted_demand': '{:,.1f}',
        'available_slots': '{:,.0f}',
        'unmet_demand': '{:,.1f}',
        'demand_to_slot_ratio': '{:.1f}x',
        'n_congested_origins': '{:,}',
        'n_congested_destinations': '{:,}',
        'demand_existing': '{:,.1f}',
        'demand_hypothetical': '{:,.1f}',
        'pct_demand_hypothetical': '{:.1f}%',
    })
)

=== Top 20 ESC Schools — Enriched View ===



,destination_school_id,total_predicted_demand,available_slots,unmet_demand,demand_to_slot_ratio,n_congested_origins,n_congested_destinations,demand_existing,demand_hypothetical,pct_demand_hypothetical
0,407210,"9,895.7",71,"9,824.7",139.4x,107,316,"4,721.2","5,174.5",52.3%
1,406357,"4,847.9",47,"4,800.9",103.1x,127,368,"2,435.4","2,412.6",49.8%
2,406740,"4,488.5",36,"4,452.5",124.7x,162,437,64.6,"4,424.0",98.6%
3,406316,"4,404.4",32,"4,372.4",137.6x,145,431,"2,122.5","2,281.9",51.8%
4,406295,"4,155.9",6,"4,149.9",692.6x,129,399,"2,484.3","1,671.5",40.2%
5,406746,"4,158.4",30,"4,128.4",138.6x,138,391,3.8,"4,154.6",99.9%
6,406341,"4,058.4",45,"4,013.4",90.2x,159,431,"1,292.3","2,766.1",68.2%
7,482073,"4,057.6",56,"4,001.6",72.5x,124,364,300.9,"3,756.7",92.6%
8,406743,"3,937.1",38,"3,899.1",103.6x,150,411,124.0,"3,813.1",96.8%
9,406294,"3,848.5",24,"3,824.5",160.4x,117,380,461.8,"3,386.7",88.0%


## 10.1 Distance-Based Unmet Demand

**Question:** Which ESC schools have high unmet demand from *nearby* origin schools feeding students to congested JHS?

**Approach:**
- Bin origin-ESC distances into **≤3 km**, **3–5 km**, and **>5 km**
- Sum predicted demand per ESC school per distance band, split by existing/hypothetical
- Compute nearby unmet demand: demand from ≤5 km origins minus available slots
- Rank ESC schools by nearby unmet demand

In [53]:
# Bin distances
cbp_congested['distance_band'] = pd.cut(
    cbp_congested['distance_km'],
    bins=[0, 3, 5, cbp_congested['distance_km'].max() + 1],
    labels=['≤3 km', '3–5 km', '>5 km'],
    right=True
)

print("Distance band distribution (congested-feeding pairs):")
print(cbp_congested['distance_band'].value_counts().sort_index())
print(f"\nMedian distance: {cbp_congested['distance_km'].median():.1f} km")
print(f"Mean distance:   {cbp_congested['distance_km'].mean():.1f} km")

Distance band distribution (congested-feeding pairs):
distance_band
≤3 km     125618
3–5 km    181784
>5 km      12115
Name: count, dtype: int64

Median distance: 3.5 km
Mean distance:   4.5 km


In [54]:
# Demand per ESC school per distance band
demand_by_dist = (
    cbp_congested
    .groupby(['destination_school_id', 'distance_band'])[REPRESENTATIVE_SCENARIO]
    .sum()
    .unstack(fill_value=0)
)
demand_by_dist.columns = [f'demand_{col}' for col in demand_by_dist.columns]

# Nearby demand (≤5 km = sum of ≤3km and 3-5km bands)
demand_by_dist['demand_nearby_5km'] = demand_by_dist['demand_≤3 km'] + demand_by_dist['demand_3–5 km']

# Merge with available slots and compute nearby unmet demand
nearby_demand = demand_by_dist.reset_index().merge(esc_slots_agg, on='destination_school_id', how='left')
nearby_demand['available_slots'] = nearby_demand['available_slots'].fillna(0)
nearby_demand['nearby_unmet_demand'] = (nearby_demand['demand_nearby_5km'] - nearby_demand['available_slots']).clip(lower=0)
nearby_demand['pct_demand_nearby'] = (
    nearby_demand['demand_nearby_5km'] /
    (nearby_demand['demand_≤3 km'] + nearby_demand['demand_3–5 km'] + nearby_demand['demand_>5 km']) * 100
)

nearby_demand = nearby_demand.sort_values('nearby_unmet_demand', ascending=False).reset_index(drop=True)

print(f"ESC schools with nearby (≤5 km) unmet demand > 0: {(nearby_demand['nearby_unmet_demand'] > 0).sum():,}")
print(f"Total nearby unmet demand: {nearby_demand['nearby_unmet_demand'].sum():,.0f}")

ESC schools with nearby (≤5 km) unmet demand > 0: 1,355
Total nearby unmet demand: 807,693


/tmp/ipykernel_2838/3522808306.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby(['destination_school_id', 'distance_band'])[REPRESENTATIVE_SCENARIO]


In [57]:
# Top 20 ESC schools by nearby unmet demand
print("=== Top 20 ESC Schools by Nearby (≤5 km) Unmet Demand ===\n")
display(
    nearby_demand[['destination_school_id', 'demand_≤3 km', 'demand_3–5 km', 'demand_>5 km',
                   'demand_nearby_5km', 'available_slots', 'nearby_unmet_demand', 'pct_demand_nearby']]
    .head(20)
    .style.format({
        'demand_≤3 km': '{:,.0f}',
        'demand_3–5 km': '{:,.0f}',
        'demand_>5 km': '{:,.0f}',
        'demand_nearby_5km': '{:,.0f}',
        'available_slots': '{:,.0f}',
        'nearby_unmet_demand': '{:,.0f}',
        'pct_demand_nearby': '{:.1f}%',
    })
)

=== Top 20 ESC Schools by Nearby (≤5 km) Unmet Demand ===



,destination_school_id,demand_≤3 km,demand_3–5 km,demand_>5 km,demand_nearby_5km,available_slots,nearby_unmet_demand,pct_demand_nearby
0,407210,"6,231","3,611",11,"9,842",71,"9,771",99.9%
1,406357,"2,240","2,548",29,"4,788",47,"4,741",99.4%
2,406740,"1,799","2,655",16,"4,454",36,"4,418",99.7%
3,406316,"2,773","1,589",14,"4,362",32,"4,330",99.7%
4,406746,"1,714","2,440",4,"4,155",30,"4,125",99.9%
5,406295,"2,633","1,483",23,"4,117",6,"4,111",99.5%
6,482073,"1,605","2,439",14,"4,044",56,"3,988",99.7%
7,406341,"1,742","2,264",19,"4,006",45,"3,961",99.5%
8,406743,"1,919","1,983",26,"3,902",38,"3,864",99.3%
9,406294,"1,755","2,087",7,"3,842",24,"3,818",99.8%


In [56]:
# Existing vs hypothetical split within the nearby (≤5 km) band
nearby_by_type = (
    cbp_congested[cbp_congested['distance_km'] <= 5]
    .groupby(['destination_school_id', 'is_hypothetical'])[REPRESENTATIVE_SCENARIO]
    .sum()
    .unstack(fill_value=0)
)
nearby_by_type = nearby_by_type.rename(columns={False: 'nearby_existing', True: 'nearby_hypothetical'})
for col in ['nearby_existing', 'nearby_hypothetical']:
    if col not in nearby_by_type.columns:
        nearby_by_type[col] = 0.0

# Merge into nearby_demand
nearby_demand = nearby_demand.merge(nearby_by_type, left_on='destination_school_id', right_index=True, how='left')
nearby_demand['nearby_pct_hypothetical'] = (
    nearby_demand['nearby_hypothetical'] / nearby_demand['demand_nearby_5km'].replace(0, np.nan) * 100
)

# Also merge n_congested_destinations from earlier
nearby_demand = nearby_demand.merge(
    n_congested_destinations.reset_index().rename(columns={'destination_school_id': 'destination_school_id'}),
    on='destination_school_id', how='left'
)

# Final enriched view — top 20
print("=== Top 20 ESC Schools — Nearby Unmet Demand (Enriched) ===\n")
display(
    nearby_demand[['destination_school_id', 'demand_nearby_5km', 'nearby_existing', 'nearby_hypothetical',
                   'nearby_pct_hypothetical', 'available_slots', 'nearby_unmet_demand',
                   'n_congested_destinations']]
    .head(20)
    .style.format({
        'demand_nearby_5km': '{:,.1f}',
        'nearby_existing': '{:,.1f}',
        'nearby_hypothetical': '{:,.1f}',
        'nearby_pct_hypothetical': '{:.1f}%',
        'available_slots': '{:,.0f}',
        'nearby_unmet_demand': '{:,.1f}',
        'n_congested_destinations': '{:,}',
    })
)

=== Top 20 ESC Schools — Nearby Unmet Demand (Enriched) ===



,destination_school_id,demand_nearby_5km,nearby_existing,nearby_hypothetical,nearby_pct_hypothetical,available_slots,nearby_unmet_demand,n_congested_destinations
0,407210,"9,841.6","4,710.3","5,174.5",52.6%,71,"9,770.6",316
1,406357,"4,788.5","2,406.3","2,412.6",50.4%,47,"4,741.5",368
2,406740,"4,454.3",48.9,"4,424.0",99.3%,36,"4,418.3",437
3,406316,"4,362.3","2,108.7","2,281.9",52.3%,32,"4,330.3",431
4,406746,"4,154.6",0.0,"4,154.6",100.0%,30,"4,124.6",391
5,406295,"4,116.6","2,461.8","1,671.5",40.6%,6,"4,110.6",399
6,482073,"4,043.9",287.2,"3,756.7",92.9%,56,"3,987.9",364
7,406341,"4,005.8","1,273.6","2,766.1",69.1%,45,"3,960.8",431
8,406743,"3,902.1",98.4,"3,813.1",97.7%,38,"3,864.1",411
9,406294,"3,841.6",454.9,"3,386.7",88.2%,24,"3,817.6",380


In [ ]:
# Export unmet demand rankings
esc_demand.to_csv(EXPORT_DIR / "esc_unmet_demand_ranking.csv", index=False)
print(f"Exported esc_unmet_demand_ranking.csv ({esc_demand.shape[0]:,} rows)")

nearby_demand.to_csv(EXPORT_DIR / "esc_nearby_unmet_demand_ranking.csv", index=False)
print(f"Exported esc_nearby_unmet_demand_ranking.csv ({nearby_demand.shape[0]:,} rows)")

print(f"\nAll Section 10 exports saved to: {EXPORT_DIR}")